# Image Captioning from Scratch
## CNN Encoder + Transformer Decoder — No Transfer Learning

**Architecture:**
- **Encoder:** Custom ResNet-style CNN (residual blocks, trained from scratch)
- **Decoder:** Transformer decoder with multi-head cross-attention over visual features
- **Training:** Label smoothing + AdamW + cosine LR schedule
- **Evaluation:** BLEU-4 on 810-image held-out test set

**Dataset:** Flickr8k (8,000 images × 5 captions) from `giorgicheishvili/caption-data`

## 1. Imports & Environment Setup

In [ ]:
import os
import re
import json
import math
import random
import pickle
import warnings
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import subprocess, sys
from pathlib import Path

if Path('/kaggle/input/caption-data').exists():
    DATASET_ROOT = Path('/kaggle/input/caption-data')
    OUTPUT_DIR   = Path('/kaggle/working')
else:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'],
        check=True
    )
    DATA_DIR = Path('data')
    DATA_DIR.mkdir(exist_ok=True)
    subprocess.run(
        [
            'kaggle', 'datasets', 'download',
            '-d', 'giorgicheishvili/caption-data',
            '-p', str(DATA_DIR),
            '--unzip',
        ],
        check=True
    )
    DATASET_ROOT = DATA_DIR
    OUTPUT_DIR   = Path('output')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dataset root : {DATASET_ROOT}')
print(f'Output dir   : {OUTPUT_DIR}')


## 2. Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
IMAGES_DIR    = DATASET_ROOT / 'Images'
CAPTIONS_FILE = DATASET_ROOT / 'captions.txt'

# ── Image ─────────────────────────────────────────────────────────────────────
IMG_SIZE      = 224          # spatial resolution fed to CNN
IMG_CHANNELS  = 3

# ── Vocabulary ────────────────────────────────────────────────────────────────
MIN_FREQ      = 3            # words appearing < MIN_FREQ times are <unk>
MAX_SEQ_LEN   = 40           # max caption length (tokens incl. <start>/<end>)

# ── CNN Encoder ───────────────────────────────────────────────────────────────
ENC_FEATURE_DIM = 512        # channels output by encoder backbone
ENC_GRID        = 7          # spatial grid size after global pool (7×7)

# ── Transformer Decoder ───────────────────────────────────────────────────────
EMBED_DIM     = 512          # token + positional embedding size
NUM_HEADS     = 8
NUM_LAYERS    = 4            # transformer decoder layers
FFN_DIM       = 2048
DROPOUT       = 0.1

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS        = 25
BATCH_SIZE    = 64
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
LABEL_SMOOTH  = 0.1
GRAD_CLIP     = 1.0

# ── Splits (Flickr8k standard) ────────────────────────────────────────────────
# 6000 train / 1000 val / 1000 test  →  test set ≈ 1000 images × 5 = 5000 caps
# Assignment asks BLEU-4 on 810 images → we sample 810 from test
N_TEST_IMAGES = 810

print('Configuration loaded.')
print(f'  Image size  : {IMG_SIZE}×{IMG_SIZE}')
print(f'  Embed dim   : {EMBED_DIM}')
print(f'  Num heads   : {NUM_HEADS}')
print(f'  Dec layers  : {NUM_LAYERS}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Epochs      : {EPOCHS}')

## 3. Data Loading & Exploration

In [ ]:
# ── Load captions.txt ─────────────────────────────────────────────────────────
# Format:  image_name,caption   (first line is header)
df = pd.read_csv(CAPTIONS_FILE)
print('Raw dataframe shape:', df.shape)
print(df.head(8))
print('\nColumns:', df.columns.tolist())

In [ ]:
# ── Normalise column names (dataset uses 'image' and 'caption') ───────────────
df.columns = [c.strip().lower() for c in df.columns]

# Some versions separate image/caption with '|'  — handle both
if 'image' not in df.columns:
    # Try pipe-delimited
    df = pd.read_csv(CAPTIONS_FILE, sep='|', header=0,
                     names=['image', 'caption_id', 'caption'])
    df = df[['image', 'caption']].copy()

df['image']   = df['image'].str.strip()
df['caption'] = df['caption'].str.strip()
df.dropna(inplace=True)

# Keep only rows whose image file exists
df = df[df['image'].apply(lambda fn: (IMAGES_DIR / fn).exists())].reset_index(drop=True)

unique_images = df['image'].nunique()
print(f'Total caption rows : {len(df):,}')
print(f'Unique images      : {unique_images:,}')
print(f'Captions/image avg : {len(df)/unique_images:.2f}')

In [ ]:
# ── Caption length distribution ───────────────────────────────────────────────
df['n_words'] = df['caption'].str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['n_words'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Caption Length Distribution')
axes[0].set_xlabel('Number of words')
axes[0].set_ylabel('Count')
axes[0].axvline(df['n_words'].mean(), color='red', linestyle='--',
                label=f"mean={df['n_words'].mean():.1f}")
axes[0].legend()

# Top 20 most frequent words
all_words = ' '.join(df['caption'].str.lower()).split()
word_freq = Counter(all_words).most_common(20)
words, counts = zip(*word_freq)
axes[1].barh(list(words)[::-1], list(counts)[::-1], color='coral')
axes[1].set_title('Top-20 Most Frequent Words')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'caption_stats.png', dpi=120)
plt.show()

print(f"Caption length  min={df['n_words'].min()}  "
      f"mean={df['n_words'].mean():.1f}  "
      f"max={df['n_words'].max()}  "
      f"95th-pct={df['n_words'].quantile(0.95):.0f}")

In [ ]:
# ── Sample images ─────────────────────────────────────────────────────────────
sample_imgs = df['image'].drop_duplicates().sample(6, random_state=SEED).tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, img_name in zip(axes.flat, sample_imgs):
    img = Image.open(IMAGES_DIR / img_name).convert('RGB')
    cap = df[df['image'] == img_name]['caption'].iloc[0]
    ax.imshow(img)
    ax.set_title(cap[:60], fontsize=8, wrap=True)
    ax.axis('off')
plt.suptitle('Sample Images with Captions', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.png', dpi=120)
plt.show()

## 4. Vocabulary

In [ ]:
class Vocabulary:
    """Simple word-level vocabulary with special tokens."""

    PAD, START, END, UNK = '<pad>', '<start>', '<end>', '<unk>'

    def __init__(self, min_freq: int = 3):
        self.min_freq  = min_freq
        self.word2idx: Dict[str, int] = {}
        self.idx2word: Dict[int, str] = {}
        self._freq:    Counter = Counter()

    # ── build ─────────────────────────────────────────────────────────────────
    def build(self, captions: List[str]) -> None:
        for cap in captions:
            for tok in self._tokenize(cap):
                self._freq[tok] += 1

        specials = [self.PAD, self.START, self.END, self.UNK]
        vocab    = specials + [w for w, c in self._freq.items() if c >= self.min_freq]
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}

    # ── helpers ───────────────────────────────────────────────────────────────
    @staticmethod
    def _tokenize(text: str) -> List[str]:
        text = text.lower().strip()
        text = re.sub(r"[^a-z0-9']", ' ', text)
        return text.split()

    def encode(self, caption: str, max_len: int) -> List[int]:
        tokens = self._tokenize(caption)
        ids = ([self.word2idx[self.START]]
               + [self.word2idx.get(t, self.word2idx[self.UNK]) for t in tokens]
               + [self.word2idx[self.END]])
        # pad / truncate
        ids = ids[:max_len]
        ids += [self.word2idx[self.PAD]] * (max_len - len(ids))
        return ids

    def decode(self, ids: List[int], skip_special: bool = True) -> str:
        words = []
        specials = {self.word2idx[s] for s in [self.PAD, self.START, self.END, self.UNK]
                    if s in self.word2idx}
        for idx in ids:
            if skip_special and idx in specials:
                if idx == self.word2idx.get(self.END):
                    break
                continue
            words.append(self.idx2word.get(idx, self.UNK))
        return ' '.join(words)

    def __len__(self) -> int:
        return len(self.word2idx)


# ── Build on training captions only (split defined below) ────────────────────
# We do a quick 80/10/10 split on unique images first
all_images = df['image'].drop_duplicates().tolist()
random.shuffle(all_images)

n  = len(all_images)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_imgs = set(all_images[:n_train])
val_imgs   = set(all_images[n_train:n_train + n_val])
test_imgs  = set(all_images[n_train + n_val:])

# Sample exactly N_TEST_IMAGES from test pool for BLEU eval
test_imgs_list = list(test_imgs)
random.shuffle(test_imgs_list)
eval_imgs = set(test_imgs_list[:N_TEST_IMAGES])

train_df = df[df['image'].isin(train_imgs)].reset_index(drop=True)
val_df   = df[df['image'].isin(val_imgs)].reset_index(drop=True)
test_df  = df[df['image'].isin(eval_imgs)].reset_index(drop=True)

print(f'Train  : {train_df["image"].nunique():,} images  {len(train_df):,} rows')
print(f'Val    : {val_df["image"].nunique():,} images  {len(val_df):,} rows')
print(f'Test   : {test_df["image"].nunique():,} images  {len(test_df):,} rows')

# Build vocabulary from TRAIN captions only
vocab = Vocabulary(min_freq=MIN_FREQ)
vocab.build(train_df['caption'].tolist())

print(f'\nVocabulary size: {len(vocab):,}')
print('Sample word→idx:', {w: vocab.word2idx[w]
      for w in ['<pad>', '<start>', '<end>', '<unk>', 'dog', 'man', 'water']
      if w in vocab.word2idx})

# Persist vocab
with open(OUTPUT_DIR / 'vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print('Vocabulary saved.')

## 5. Dataset & DataLoader

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
# Statistics computed on ImageNet are NOT used — we use rough natural-image norms
# since we have no pretrained weights to align to.
_MEAN = [0.5, 0.5, 0.5]
_STD  = [0.5, 0.5, 0.5]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])


class Flickr8kDataset(Dataset):
    """
    Returns (image_tensor, input_ids, target_ids) where:
      input_ids  = [<start>, w1, w2, ..., wN, <pad> ...]
      target_ids = [w1, w2, ..., wN, <end>, <pad> ...]
    We randomly pick ONE of the 5 captions per image per epoch.
    """

    def __init__(self,
                 dataframe: pd.DataFrame,
                 images_dir: Path,
                 vocab: Vocabulary,
                 max_len: int,
                 transform=None,
                 training: bool = False):
        self.df         = dataframe
        self.images_dir = images_dir
        self.vocab      = vocab
        self.max_len    = max_len
        self.transform  = transform
        self.training   = training

        # Group captions per image for random sampling during training
        self.image_list = dataframe['image'].drop_duplicates().tolist()
        self.img2caps   = dataframe.groupby('image')['caption'].apply(list).to_dict()

        if not training:
            # Fixed order: expand so each (image, caption) pair is one sample
            self.pairs = list(zip(dataframe['image'], dataframe['caption']))

    def __len__(self) -> int:
        if self.training:
            return len(self.image_list)
        return len(self.pairs)

    def __getitem__(self, idx: int):
        if self.training:
            img_name = self.image_list[idx]
            caption  = random.choice(self.img2caps[img_name])
        else:
            img_name, caption = self.pairs[idx]

        # Load image
        img = Image.open(self.images_dir / img_name).convert('RGB')
        if self.transform:
            img = self.transform(img)

        # Encode caption
        ids = self.vocab.encode(caption, self.max_len)
        ids = torch.tensor(ids, dtype=torch.long)

        # Shift: input = ids[:-1], target = ids[1:]
        inp_ids = ids[:-1]    # <start> w1 w2 ...  (length MAX_SEQ_LEN - 1)
        tgt_ids = ids[1:]     # w1  w2 ... <end>   (length MAX_SEQ_LEN - 1)

        return img, inp_ids, tgt_ids


train_dataset = Flickr8kDataset(train_df, IMAGES_DIR, vocab, MAX_SEQ_LEN,
                                 train_transform, training=True)
val_dataset   = Flickr8kDataset(val_df,   IMAGES_DIR, vocab, MAX_SEQ_LEN,
                                 eval_transform, training=False)
test_dataset  = Flickr8kDataset(test_df,  IMAGES_DIR, vocab, MAX_SEQ_LEN,
                                 eval_transform, training=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')
print(f'Test  batches: {len(test_loader)}')

# Sanity-check one batch
imgs, inp, tgt = next(iter(train_loader))
print(f'\nBatch shapes  img={imgs.shape}  inp={inp.shape}  tgt={tgt.shape}')
print('Sample decoded input :', vocab.decode(inp[0].tolist(), skip_special=False))
print('Sample decoded target:', vocab.decode(tgt[0].tolist(), skip_special=False))

## 6. Model Architecture

### 6.1 CNN Encoder (ResNet-style, from scratch)

In [ ]:
class ResidualBlock(nn.Module):
    """
    Standard ResNet bottleneck block (1×1 → 3×3 → 1×1).
    Optionally downsamples the identity shortcut.
    """
    expansion = 4

    def __init__(self, in_ch: int, mid_ch: int, stride: int = 1,
                 downsample: Optional[nn.Module] = None):
        super().__init__()
        out_ch = mid_ch * self.expansion

        self.conv1 = nn.Conv2d(in_ch, mid_ch, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(mid_ch)

        self.conv2 = nn.Conv2d(mid_ch, mid_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(mid_ch)

        self.conv3 = nn.Conv2d(mid_ch, out_ch, 1, bias=False)
        self.bn3   = nn.BatchNorm2d(out_ch)

        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


class CNNEncoder(nn.Module):
    """
    Custom ResNet-50-like encoder trained from scratch.
    Output: (B, num_pixels, enc_dim)  where num_pixels = grid_h × grid_w

    Layers:
        stem   → conv 7×7/2 → BN → ReLU → MaxPool/2
        layer1 → 3 bottleneck blocks, 64 mid-ch  (output 256-ch)
        layer2 → 4 bottleneck blocks, 128 mid-ch (output 512-ch, stride 2)
        layer3 → 6 bottleneck blocks, 256 mid-ch (output 1024-ch, stride 2)
        layer4 → 3 bottleneck blocks, 512 mid-ch (output 2048-ch, stride 2)
        proj   → 1×1 conv 2048→enc_dim
        → output grid: 7×7 for 224×224 input
    """

    def __init__(self, enc_dim: int = 512):
        super().__init__()
        self.enc_dim = enc_dim
        self.in_ch   = 64

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )

        # Residual stages
        self.layer1 = self._make_stage(64,  blocks=3, stride=1)
        self.layer2 = self._make_stage(128, blocks=4, stride=2)
        self.layer3 = self._make_stage(256, blocks=6, stride=2)
        self.layer4 = self._make_stage(512, blocks=3, stride=2)

        # Project 2048 → enc_dim
        self.proj = nn.Sequential(
            nn.Conv2d(2048, enc_dim, 1, bias=False),
            nn.BatchNorm2d(enc_dim),
            nn.ReLU(inplace=True),
        )

        self._init_weights()

    def _make_stage(self, mid_ch: int, blocks: int, stride: int) -> nn.Sequential:
        out_ch = mid_ch * ResidualBlock.expansion
        downsample = None
        if stride != 1 or self.in_ch != out_ch:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        layers = [ResidualBlock(self.in_ch, mid_ch, stride, downsample)]
        self.in_ch = out_ch
        for _ in range(1, blocks):
            layers.append(ResidualBlock(self.in_ch, mid_ch))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, H, W)
        x = self.stem(x)       # (B, 64,  H/4,  W/4)
        x = self.layer1(x)     # (B, 256, H/4,  W/4)
        x = self.layer2(x)     # (B, 512, H/8,  W/8)
        x = self.layer3(x)     # (B, 1024, H/16, W/16)
        x = self.layer4(x)     # (B, 2048, H/32, W/32)
        x = self.proj(x)       # (B, enc_dim, H/32, W/32)  → 7×7 for 224 input

        B, C, Hg, Wg = x.shape
        x = x.flatten(2).permute(0, 2, 1)   # (B, num_pixels, enc_dim)
        return x


# Quick shape test
enc_test = CNNEncoder(ENC_FEATURE_DIM).to(DEVICE)
dummy    = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
enc_out  = enc_test(dummy)
print(f'Encoder output shape: {enc_out.shape}')   # expect (2, 49, 512)
n_params = sum(p.numel() for p in enc_test.parameters())
print(f'Encoder params: {n_params/1e6:.2f}M')
del enc_test, dummy, enc_out

### 6.2 Transformer Decoder

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)   # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class TransformerDecoder(nn.Module):
    """
    Standard Transformer decoder that cross-attends over visual features.

    Architecture:
        token embed → positional encoding
        N × DecoderLayer:
            Masked Self-Attention (causal)
            Cross-Attention over encoder output
            FFN
        LayerNorm → linear projection to vocab
    """

    def __init__(self,
                 vocab_size: int,
                 enc_dim:    int,
                 embed_dim:  int,
                 num_heads:  int,
                 num_layers: int,
                 ffn_dim:    int,
                 max_len:    int,
                 dropout:    float,
                 pad_idx:    int):
        super().__init__()
        self.embed_dim  = embed_dim
        self.pad_idx    = pad_idx

        # Visual feature projection (enc_dim → embed_dim if different)
        self.vis_proj = (nn.Linear(enc_dim, embed_dim)
                         if enc_dim != embed_dim else nn.Identity())

        # Token embedding + positional encoding
        self.token_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_enc     = PositionalEncoding(embed_dim, max_len, dropout)

        # Decoder stack
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,  # pre-norm
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.norm        = nn.LayerNorm(embed_dim)

        # Output head
        self.head = nn.Linear(embed_dim, vocab_size)

        # Weight tying: share embedding → output projection weights
        self.head.weight = self.token_embed.weight

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
        if isinstance(self.vis_proj, nn.Linear):
            nn.init.xavier_uniform_(self.vis_proj.weight)
            nn.init.zeros_(self.vis_proj.bias)

    def forward(self,
                enc_out:   torch.Tensor,   # (B, num_pix, enc_dim)
                tgt_ids:   torch.Tensor,   # (B, T)
                ) -> torch.Tensor:         # (B, T, vocab_size)

        B, T = tgt_ids.shape

        # Causal mask (upper triangular)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(
            T, device=tgt_ids.device)     # (T, T)

        # Padding mask
        tgt_key_padding = (tgt_ids == self.pad_idx)  # (B, T)  True = ignore

        # Embed tokens
        tok = self.token_embed(tgt_ids) * math.sqrt(self.embed_dim)  # (B, T, E)
        tok = self.pos_enc(tok)

        # Project visual features
        mem = self.vis_proj(enc_out)   # (B, num_pix, E)

        # Decode
        out = self.transformer(
            tgt=tok, memory=mem,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding,
        )                             # (B, T, E)
        out = self.norm(out)
        return self.head(out)         # (B, T, vocab_size)


# Shape test
dec_test = TransformerDecoder(
    vocab_size=1000, enc_dim=ENC_FEATURE_DIM, embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS, num_layers=NUM_LAYERS, ffn_dim=FFN_DIM,
    max_len=MAX_SEQ_LEN, dropout=DROPOUT,
    pad_idx=0
).to(DEVICE)
dummy_enc = torch.randn(2, 49, ENC_FEATURE_DIM).to(DEVICE)
dummy_tok = torch.randint(0, 1000, (2, MAX_SEQ_LEN - 1)).to(DEVICE)
dec_out   = dec_test(dummy_enc, dummy_tok)
print(f'Decoder output shape: {dec_out.shape}')   # (2, 39, 1000)
n_params = sum(p.numel() for p in dec_test.parameters())
print(f'Decoder params: {n_params/1e6:.2f}M')
del dec_test, dummy_enc, dummy_tok, dec_out

### 6.3 Full Model

In [ ]:
class ImageCaptioningModel(nn.Module):
    """End-to-end image captioning model (CNN Encoder + Transformer Decoder)."""

    def __init__(self, vocab_size: int, pad_idx: int):
        super().__init__()
        self.encoder = CNNEncoder(enc_dim=ENC_FEATURE_DIM)
        self.decoder = TransformerDecoder(
            vocab_size=vocab_size,
            enc_dim=ENC_FEATURE_DIM,
            embed_dim=EMBED_DIM,
            num_heads=NUM_HEADS,
            num_layers=NUM_LAYERS,
            ffn_dim=FFN_DIM,
            max_len=MAX_SEQ_LEN,
            dropout=DROPOUT,
            pad_idx=pad_idx,
        )

    def forward(self, images: torch.Tensor, inp_ids: torch.Tensor) -> torch.Tensor:
        enc_out = self.encoder(images)          # (B, num_pix, enc_dim)
        logits  = self.decoder(enc_out, inp_ids)  # (B, T, vocab_size)
        return logits

    @torch.no_grad()
    def generate_greedy(self,
                        images:    torch.Tensor,
                        start_idx: int,
                        end_idx:   int,
                        max_len:   int = 40) -> List[List[int]]:
        """Greedy decoding — used for quick val accuracy estimate."""
        self.eval()
        B = images.size(0)
        enc_out = self.encoder(images)

        # Start token for every item in batch
        tokens = torch.full((B, 1), start_idx, dtype=torch.long, device=images.device)
        finished = torch.zeros(B, dtype=torch.bool, device=images.device)
        sequences = [[] for _ in range(B)]

        for _ in range(max_len - 1):
            logits = self.decoder(enc_out, tokens)   # (B, t, V)
            next_tok = logits[:, -1, :].argmax(-1)   # (B,)
            for i in range(B):
                if not finished[i]:
                    sequences[i].append(next_tok[i].item())
                    if next_tok[i].item() == end_idx:
                        finished[i] = True
            tokens = torch.cat([tokens, next_tok.unsqueeze(1)], dim=1)
            if finished.all():
                break

        return sequences

    @torch.no_grad()
    def generate_beam(self,
                      image:     torch.Tensor,   # (1, C, H, W)
                      start_idx: int,
                      end_idx:   int,
                      beam_size: int = 5,
                      max_len:   int = 40) -> List[int]:
        """Beam search decoding for BLEU-4 evaluation (single image)."""
        self.eval()
        enc_out = self.encoder(image)   # (1, num_pix, enc_dim)

        # Each beam: (score, token_list)
        beams  = [(0.0, [start_idx])]
        done   = []

        for _ in range(max_len - 1):
            all_candidates = []
            for score, seq in beams:
                if seq[-1] == end_idx:
                    done.append((score, seq))
                    continue
                t_ids = torch.tensor([seq], dtype=torch.long, device=image.device)
                # Expand enc_out to match beam (already 1, just reuse)
                logits  = self.decoder(enc_out, t_ids)          # (1, t, V)
                log_prob = F.log_softmax(logits[0, -1], dim=-1) # (V,)
                top_val, top_idx = log_prob.topk(beam_size)
                for val, idx in zip(top_val, top_idx):
                    all_candidates.append((score + val.item(), seq + [idx.item()]))

            if not all_candidates:
                break
            beams = sorted(all_candidates, key=lambda x: x[0], reverse=True)[:beam_size]

        done += beams
        best_score, best_seq = max(done, key=lambda x: x[0] / max(len(x[1]), 1))
        return best_seq


PAD_IDX   = vocab.word2idx[Vocabulary.PAD]
START_IDX = vocab.word2idx[Vocabulary.START]
END_IDX   = vocab.word2idx[Vocabulary.END]

model = ImageCaptioningModel(vocab_size=len(vocab), pad_idx=PAD_IDX).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters   : {total_params/1e6:.2f}M')
print(f'Trainable params   : {train_params/1e6:.2f}M')

## 7. Loss, Optimiser & Scheduler

In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross-entropy with label smoothing, ignoring PAD tokens.
    Smoothing distributes `epsilon` probability mass uniformly over vocab.
    """

    def __init__(self, vocab_size: int, pad_idx: int, smoothing: float = 0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_idx    = pad_idx
        self.smoothing  = smoothing
        self.confidence = 1.0 - smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        logits  : (B*T, V)
        targets : (B*T,)
        """
        B_T, V = logits.shape
        log_prob = F.log_softmax(logits, dim=-1)

        # Smooth target distribution
        smooth_dist = torch.full_like(log_prob, self.smoothing / (V - 2))  # -2: pad + target
        smooth_dist.scatter_(1, targets.unsqueeze(1), self.confidence)
        smooth_dist[:, self.pad_idx] = 0.0

        loss = -(smooth_dist * log_prob).sum(dim=-1)

        # Mask padding
        non_pad = (targets != self.pad_idx)
        loss    = loss[non_pad]
        return loss.mean()


criterion = LabelSmoothingCrossEntropy(len(vocab), PAD_IDX, LABEL_SMOOTH)

# ── Optimiser ─────────────────────────────────────────────────────────────────
# Separate param groups: higher LR for decoder, lower for encoder
enc_params = list(model.encoder.parameters())
dec_params = list(model.decoder.parameters())

optimizer = torch.optim.AdamW([
    {'params': enc_params, 'lr': LR * 0.5},   # encoder needs gentler updates
    {'params': dec_params, 'lr': LR},
], weight_decay=WEIGHT_DECAY)

# ── Scheduler: cosine annealing ───────────────────────────────────────────────
total_steps = EPOCHS * len(train_loader)
warmup_steps = int(0.05 * total_steps)          # 5% warmup

def lr_lambda(step: int) -> float:
    if step < warmup_steps:
        return float(step) / max(warmup_steps, 1)
    progress = float(step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f'Optimiser  : AdamW  (enc LR={LR*0.5:.2e}, dec LR={LR:.2e})')
print(f'Scheduler  : cosine annealing + {warmup_steps} warmup steps')
print(f'Total steps: {total_steps:,}')

## 8. Training & Validation Loop

In [ ]:
def token_accuracy(logits: torch.Tensor, targets: torch.Tensor, pad_idx: int) -> float:
    """Proportion of non-pad tokens predicted correctly."""
    preds   = logits.argmax(-1)          # (B, T)
    mask    = (targets != pad_idx)
    correct = ((preds == targets) & mask).sum().item()
    total   = mask.sum().item()
    return correct / max(total, 1)


def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    for imgs, inp_ids, tgt_ids in loader:
        imgs    = imgs.to(DEVICE, non_blocking=True)
        inp_ids = inp_ids.to(DEVICE, non_blocking=True)
        tgt_ids = tgt_ids.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
            logits = model(imgs, inp_ids)                  # (B, T, V)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V),
                             tgt_ids.reshape(B * T))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        acc = token_accuracy(logits.detach(), tgt_ids, PAD_IDX)
        total_loss += loss.item()
        total_acc  += acc
        n_batches  += 1

    return total_loss / n_batches, total_acc / n_batches


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, total_acc, n_batches = 0.0, 0.0, 0

    for imgs, inp_ids, tgt_ids in loader:
        imgs    = imgs.to(DEVICE, non_blocking=True)
        inp_ids = inp_ids.to(DEVICE, non_blocking=True)
        tgt_ids = tgt_ids.to(DEVICE, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
            logits = model(imgs, inp_ids)
            B, T, V = logits.shape
            loss = criterion(logits.reshape(B * T, V),
                             tgt_ids.reshape(B * T))

        acc = token_accuracy(logits, tgt_ids, PAD_IDX)
        total_loss += loss.item()
        total_acc  += acc
        n_batches  += 1

    return total_loss / n_batches, total_acc / n_batches


print('Training loop functions defined.')

In [ ]:
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   [],
    'lr': [],
}

best_val_loss = float('inf')
CHECKPOINT    = OUTPUT_DIR / 'best_model.pth'

print(f'Starting training for {EPOCHS} epochs...\n')
print(f'{"Epoch":>5}  {"TrLoss":>8}  {"TrAcc":>7}  '
      f'{"VlLoss":>8}  {"VlAcc":>7}  {"LR":>10}')
print('-' * 60)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer,
                                      scheduler, criterion, scaler)
    vl_loss, vl_acc = validate(model, val_loader, criterion)
    current_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss)
    history['val_acc'].append(vl_acc)
    history['lr'].append(current_lr)

    # Save best checkpoint
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save({
            'epoch':      epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_loss':   vl_loss,
            'vocab_size': len(vocab),
            'config': {
                'enc_dim': ENC_FEATURE_DIM, 'embed_dim': EMBED_DIM,
                'num_heads': NUM_HEADS, 'num_layers': NUM_LAYERS,
                'ffn_dim': FFN_DIM, 'dropout': DROPOUT,
                'max_len': MAX_SEQ_LEN,
            },
        }, CHECKPOINT)
        tag = ' ← best'
    else:
        tag = ''

    print(f'{epoch:>5}  {tr_loss:>8.4f}  {tr_acc:>7.4f}  '
          f'{vl_loss:>8.4f}  {vl_acc:>7.4f}  {current_lr:>10.2e}{tag}')

print('\nTraining complete.')
print(f'Best val loss: {best_val_loss:.4f}  (saved to {CHECKPOINT})')

## 9. Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(epochs_range, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs_range, history['val_loss'],   label='Val',   color='coral')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Label-Smoothed CE Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, [a * 100 for a in history['train_acc']],
             label='Train', color='steelblue')
axes[1].plot(epochs_range, [a * 100 for a in history['val_acc']],
             label='Val',   color='coral')
axes[1].set_title('Token Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

# LR schedule
axes[2].plot(epochs_range, history['lr'], color='mediumseagreen')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=120)
plt.show()

print(f'Final train loss: {history["train_loss"][-1]:.4f} | '
      f'acc: {history["train_acc"][-1]*100:.2f}%')
print(f'Final val   loss: {history["val_loss"][-1]:.4f} | '
      f'acc: {history["val_acc"][-1]*100:.2f}%')

## 10. BLEU-4 Evaluation on 810 Test Images

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]} '
      f'(val_loss={ckpt["val_loss"]:.4f})')

In [ ]:
def compute_bleu4(model,
                  test_df: pd.DataFrame,
                  images_dir: Path,
                  vocab: Vocabulary,
                  transform,
                  beam_size: int = 5,
                  max_len: int = 40) -> float:
    """
    Compute corpus BLEU-4 over all unique images in test_df.
    References = all 5 captions per image.
    Hypothesis = beam-search caption.
    """
    model.eval()
    smoothing_fn = SmoothingFunction().method1

    img2refs = test_df.groupby('image')['caption'].apply(list).to_dict()
    unique   = list(img2refs.keys())

    all_refs  = []
    all_hyps  = []

    for i, img_name in enumerate(unique):
        img = Image.open(images_dir / img_name).convert('RGB')
        img_t = transform(img).unsqueeze(0).to(DEVICE)  # (1, C, H, W)

        # Beam search
        token_ids = model.generate_beam(
            img_t, START_IDX, END_IDX, beam_size=beam_size, max_len=max_len
        )
        hyp = vocab.decode(token_ids, skip_special=True).split()

        # References: tokenise each of the 5 captions
        refs = [vocab._tokenize(cap) for cap in img2refs[img_name]]

        all_hyps.append(hyp)
        all_refs.append(refs)

        if (i + 1) % 100 == 0:
            print(f'  Evaluated {i+1}/{len(unique)} images...')

    bleu4 = corpus_bleu(all_refs, all_hyps,
                        smoothing_function=smoothing_fn)
    return bleu4, all_refs, all_hyps, unique


print(f'Running BLEU-4 evaluation on {N_TEST_IMAGES} test images '
      f'(beam_size=5)...')
bleu4, all_refs, all_hyps, test_image_names = compute_bleu4(
    model, test_df, IMAGES_DIR, vocab, eval_transform, beam_size=5
)
print(f'\n★ BLEU-4 Score: {bleu4:.4f}  ({bleu4*100:.2f}%)')

## 11. Qualitative Analysis

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

# Per-image BLEU-4
smoothing_fn = SmoothingFunction().method1
per_image_bleu = [
    sentence_bleu(refs, hyp, smoothing_function=smoothing_fn)
    for refs, hyp in zip(all_refs, all_hyps)
]

# Sort by BLEU score
sorted_by_bleu = sorted(zip(per_image_bleu, test_image_names, all_refs, all_hyps),
                        key=lambda x: x[0], reverse=True)

# Top-5 successes
TOP_K = 5
best_examples  = sorted_by_bleu[:TOP_K]
worst_examples = sorted_by_bleu[-TOP_K:]


def plot_examples(examples, title, filename):
    fig, axes = plt.subplots(1, TOP_K, figsize=(18, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for ax, (bleu, img_name, refs, hyp) in zip(axes, examples):
        img = Image.open(IMAGES_DIR / img_name).convert('RGB')
        ax.imshow(img)
        ref_text = refs[0][:10]   # first ref, truncated for display
        hyp_text = hyp[:10]
        ax.set_title(
            f'BLEU-4: {bleu:.3f}\n'
            f'Gen: {", ".join(hyp_text)}\n'
            f'Ref: {", ".join(ref_text)}',
            fontsize=7, wrap=True
        )
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=120, bbox_inches='tight')
    plt.show()


plot_examples(best_examples,  'Top-5 Best Captions',  'best_captions.png')
plot_examples(worst_examples, 'Top-5 Worst Captions', 'worst_captions.png')

In [ ]:
# ── Per-image BLEU distribution ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(per_image_bleu, bins=40, color='mediumpurple', edgecolor='white', alpha=0.85)
ax.axvline(bleu4, color='red', linestyle='--',
           label=f'Corpus BLEU-4 = {bleu4:.4f}')
ax.set_title('Per-Image BLEU-4 Distribution (Test Set)')
ax.set_xlabel('BLEU-4')
ax.set_ylabel('Count')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bleu_distribution.png', dpi=120)
plt.show()

print(f'Per-image BLEU-4   mean={np.mean(per_image_bleu):.4f}  '
      f'median={np.median(per_image_bleu):.4f}  '
      f'std={np.std(per_image_bleu):.4f}')

## 12. Summary & Saved Artefacts

In [ ]:
# Save full history and results
results = {
    'bleu4':           bleu4,
    'best_val_loss':   best_val_loss,
    'final_train_loss': history['train_loss'][-1],
    'final_val_loss':   history['val_loss'][-1],
    'final_train_acc':  history['train_acc'][-1],
    'final_val_acc':    history['val_acc'][-1],
    'n_test_images':   N_TEST_IMAGES,
    'vocab_size':      len(vocab),
    'total_params_M':  total_params / 1e6,
    'history':         history,
}

with open(OUTPUT_DIR / 'results.json', 'w') as f:
    json.dump(
        {k: v for k, v in results.items() if k != 'history'},
        f, indent=2
    )

print('=' * 60)
print('        FINAL RESULTS SUMMARY')
print('=' * 60)
print(f'  Architecture    : ResNet-50-style CNN + Transformer Decoder')
print(f'  Transfer learn  : NONE (trained fully from scratch)')
print(f'  Total params    : {total_params/1e6:.2f}M')
print(f'  Vocab size      : {len(vocab):,}')
print(f'  Train images    : {train_df["image"].nunique():,}')
print(f'  Test images     : {N_TEST_IMAGES:,}')
print('-' * 60)
print(f'  Final train loss: {history["train_loss"][-1]:.4f}')
print(f'  Final val   loss: {history["val_loss"][-1]:.4f}')
print(f'  Final train acc : {history["train_acc"][-1]*100:.2f}%')
print(f'  Final val   acc : {history["val_acc"][-1]*100:.2f}%')
print(f'  Best val   loss : {best_val_loss:.4f}')
print('-' * 60)
print(f'  ★ BLEU-4 (beam=5, {N_TEST_IMAGES} imgs): {bleu4:.4f}  '
      f'({bleu4*100:.2f}%)')
print('=' * 60)

print('\nSaved artefacts:')
for p in sorted(OUTPUT_DIR.glob('*')):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<30} {size_kb:>8.1f} KB')